<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/gru/gru-bit-parity-classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# -------------------------------
# Simple GRU Cell
# -------------------------------
class SimpleGRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        # Update gate weights – controls how much of the previous hidden state
        # vs. the candidate hidden state contributes to the new hidden state
        self.input_to_update = nn.Linear(input_size, hidden_size)
        self.hidden_to_update = nn.Linear(hidden_size, hidden_size)

        # Reset gate weights – controls how much of the previous hidden state
        # is used to compute the candidate hidden state
        self.input_to_reset = nn.Linear(input_size, hidden_size)
        self.hidden_to_reset = nn.Linear(hidden_size, hidden_size)

        # Candidate hidden state weights – used to compute a new candidate
        # hidden state based on the current input and (reset-modulated) hidden state
        self.input_to_candidate = nn.Linear(input_size, hidden_size)
        self.hidden_to_candidate = nn.Linear(hidden_size, hidden_size)

    def forward(self, current_input, prev_hidden):
        # current_input: (batch_size, input_size)
        # prev_hidden: (batch_size, hidden_size)

        # 1. Compute update gate
        update_gate = torch.sigmoid(
            self.input_to_update(current_input) +
            self.hidden_to_update(prev_hidden)
        )

        # 2. Compute reset gate
        reset_gate = torch.sigmoid(
            self.input_to_reset(current_input) +
            self.hidden_to_reset(prev_hidden)
        )

        # 3. Compute candidate hidden state
        reset_hidden = reset_gate * prev_hidden
        candidate_hidden = torch.tanh(
            self.input_to_candidate(current_input) +
            self.hidden_to_candidate(reset_hidden)
        )

        # 4. Compute new hidden state
        new_hidden = (1 - update_gate) * prev_hidden + update_gate * candidate_hidden

        return new_hidden

# -------------------------------
# GRU Sequence Wrapper
# -------------------------------
class SimpleGRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = SimpleGRUCell(input_size, hidden_size)

    def forward(self, x, h_0=None):
        batch_size, seq_len, _ = x.size()

        if h_0 is None:
            h_t = torch.zeros(batch_size, self.hidden_size, device=x.device)
        else:
            h_t = h_0.squeeze(0)

        outputs = []

        for t in range(seq_len):
            h_t = self.cell(x[:, t, :], h_t)
            outputs.append(h_t.unsqueeze(1))  # (batch, 1, hidden)

        output = torch.cat(outputs, dim=1)           # (batch, seq_len, hidden)
        h_n = h_t.unsqueeze(0)                        # (1, batch, hidden)

        return output, h_n

# -------------------------------
# Classification Model
# -------------------------------
class GRUClassifier(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.gru = SimpleGRU(input_size, hidden_size)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h_n = self.gru(x)
        h_n = h_n.squeeze(0)  # (batch, hidden)
        out = self.fc(h_n)
        return torch.sigmoid(out).squeeze(1)

# -------------------------------
# Data Generator (Pattern-Based)
# -------------------------------
def generate_batch(batch_size, seq_len):
    X = []
    y = []
    for _ in range(batch_size):
        seq = [random.choice([0, 1]) for _ in range(seq_len)]
        label = int(sum(seq) > seq_len // 2)
        x_seq = torch.tensor(seq, dtype=torch.float32).unsqueeze(-1)  # (seq_len, 1)
        X.append(x_seq)
        y.append(label)
    return torch.stack(X), torch.tensor(y, dtype=torch.float32)

# -------------------------------
# Training Loop
# -------------------------------
def train():
    input_size = 1
    hidden_size = 16
    seq_len = 200
    batch_size = 32
    epochs = 5000
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = GRUClassifier(input_size, hidden_size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = nn.BCELoss()

    for epoch in range(1, epochs + 1):
        model.train()
        X_batch, y_batch = generate_batch(batch_size, seq_len)
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        preds = model(X_batch)
        loss = loss_fn(preds, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            predicted = (preds > 0.5).float()
            acc = (predicted == y_batch).float().mean().item()

        print(f"Epoch {epoch:02d} | Loss: {loss.item():.4f} | Accuracy: {acc * 100:.2f}%")

        if acc == 1.0:
            print("✅ Achieved 100% accuracy.")
            break

if __name__ == "__main__":
    train()

Epoch 01 | Loss: 0.6704 | Accuracy: 65.62%
Epoch 02 | Loss: 0.6806 | Accuracy: 56.25%
Epoch 03 | Loss: 0.6797 | Accuracy: 56.25%
Epoch 04 | Loss: 0.7309 | Accuracy: 46.88%
Epoch 05 | Loss: 0.7381 | Accuracy: 43.75%
Epoch 06 | Loss: 0.6856 | Accuracy: 53.12%
Epoch 07 | Loss: 0.6920 | Accuracy: 50.00%
Epoch 08 | Loss: 0.6831 | Accuracy: 68.75%
Epoch 09 | Loss: 0.6841 | Accuracy: 65.62%
Epoch 10 | Loss: 0.6763 | Accuracy: 62.50%
Epoch 11 | Loss: 0.6939 | Accuracy: 56.25%
Epoch 12 | Loss: 0.6916 | Accuracy: 53.12%
Epoch 13 | Loss: 0.6960 | Accuracy: 46.88%
Epoch 14 | Loss: 0.6930 | Accuracy: 50.00%
Epoch 15 | Loss: 0.6946 | Accuracy: 46.88%
Epoch 16 | Loss: 0.6859 | Accuracy: 53.12%
Epoch 17 | Loss: 0.6739 | Accuracy: 62.50%
Epoch 18 | Loss: 0.6717 | Accuracy: 59.38%
Epoch 19 | Loss: 0.6783 | Accuracy: 62.50%
Epoch 20 | Loss: 0.6561 | Accuracy: 59.38%
Epoch 21 | Loss: 0.6698 | Accuracy: 56.25%
Epoch 22 | Loss: 0.6772 | Accuracy: 53.12%
Epoch 23 | Loss: 0.7427 | Accuracy: 40.62%
Epoch 24 | 